In [ ]:
## load PFCLM output and compare to Ameriflux data

import parflow as pf
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from glob import glob
import pandas as pd
import parflow as pf
from pathlib import Path

from parflow import Run
import parflow.tools.hydrology as hydro


import hf_hydrodata as hf

# swap to your email and PIN; you may not this on Verde
## [register PIN here](https://hydrogen.princeton.edu/pin)
#hf.gridded.register_api_pin("<your_email>", "<your_pin>")


In [ ]:

# set the path to the data and the date range: this should be your values for year and site
start = "2016-10-01"
end = "2017-10-01"
static_write_dir = "runs/snow/Schofield_Pass_WY2017"   # the run directory from Notebooks 1 and 2
base = static_write_dir 
site_id = 'US-Ne2'

print('working directory: ', base)


In [ ]:

# load in the run information from the PFIDB file
run_name = 'pfclm_sc'
run = Run.from_definition(f'{base}/{run_name}.pfidb')
data = run.data_accessor
nx = data.shape[2]
ny = data.shape[1]
nz = data.shape[0]
dx = data.dx
dy = data.dy
dz = data.dz


# intialize data and time arrays
data    = np.zeros([8,8760])
time    = np.zeros([8760])

# load in output files
file_in = f"{base}/{run_name}.out.00001.nc"
print(file_in)
pf_out = xr.open_mfdataset(file_in)
print(pf_out.head())

file_in = f"{base}/{run_name}.out.CLM.00001.nc"
print(file_in)
clm_out = xr.open_mfdataset(file_in)
print(clm_out.head())


In [ ]:

# load the forcing file into a Pandas dataframe

# load forcing, single column, variables are
#DSWR:  Downward Visible or Short-Wave radiation [W/m2]
#DLWR:  Downward Infa-Red or Long-Wave radiation [W/m2]
#APCP:  Precipitation rate [mm/s]
#Temp:  Air temperature [K]
#UGRD:  West-to-East or U-component of wind [m/s]
#VGRD:  South-to-North or V-component of wind [m/s]
#Press: Atmospheric Pressure [pa]
#SPFH:  Water-vapor specific humidity [kg/kg]

# load the forcing file into a Pandas dataframe
# we could load this file directly, as is shown in the commented-out line, but we can also
# use the key database and load it dynamically as shown

file = base+'/'+run.Solver.CLM.MetFileName
print(file)

# 1D forcing files are text columns not technically CSVs but this is an easy 
# way to load
forcing_df = pd.read_csv(file,sep=' ',header=None,engine='python') 
# add PF-CLM variable names and units 
forcing_df.columns = ["DSWR [W/m2]", "DLWR [W/m2]", "APCP [mm/s]", "Temp [K]", "UGRD [m/s]","VGRD [m/s]","Press [pa]","SPFH [kg/kg]"]
print(forcing_df.head())


In [ ]:
## make a plot of the output from the PFCLM run

res_sat = np.zeros(8760)
res_sat[:] = run.Solver.CLM.ResSat
porosity = run.Geom.soil.Porosity.Value      # the column has two units, soil and geology; the top cell is soil
mannings = run.Mannings.Geom.domain.Value
slope = run.TopoSlopesX.Geom.domain.Value


def calc_runoff(pf_out):
    # Manning's equation on the ponded depth at the top cell: runoff per unit width [m2/h]
    return (np.sqrt(slope)/mannings) * np.maximum(pf_out['pressure'][:,9,0,0],0.0)**(5.0/3.0)

runoff = calc_runoff(pf_out)
# the same runoff as a depth over the cell [mm/h], to compare with precipitation
runoff_mmh = runoff / dx * 1000.0
print('total runoff [mm]: ', float(runoff_mmh.sum()), '  total precipitation [mm]: ', float(forcing_df['APCP [mm/s]'].sum()*3600))


# Plot the soil moisture at the top cell with precip and runoff for our PFCLM run
fig, ax = plt.subplots()
ax2 = ax.twinx()
ax.plot(pf_out['saturation'][:,9,0,0]*100*porosity, label='Soil Moisture 5cm',color='red') 
ax2.plot(runoff, label='Runoff [m2/h]',color='green') 
ax.plot(forcing_df['APCP [mm/s]']*3600, label='Precipitation [mm/h]',color='blue')
ax.set_xlabel('Time [hr]')  
ax.set_ylabel('Water Content and Precipitation')  
ax2.set_ylabel('Runoff') 
plt.legend()
plt.show()



In [ ]:
# load in the run information from the PFIDB file
run_name = 'pfclm_sc'
run = Run.from_definition(f'{base}/{run_name}.pfidb')

# change values and re-run parflow; as written each line keeps the value from the prior run
# the column has two units: soil (top 2 m) and geology (below); change one or both
run.Geom.soil.Porosity.Value    = 1.0 * run.Geom.soil.Porosity.Value
run.Geom.geology.Porosity.Value = 1.0 * run.Geom.geology.Porosity.Value

run.Geom.soil.Perm.Value    = 1.0 * run.Geom.soil.Perm.Value       # units of K are m/h
run.Geom.geology.Perm.Value = 1.0 * run.Geom.geology.Perm.Value

wtd_m = run.Patch.z_lower.BCPressure.alltime.Value   # e.g. -1.0 puts the water table 1 m below land surface
run.Patch.z_lower.BCPressure.alltime.Value = wtd_m
run.Geom.domain.ICPressure.Value           = wtd_m   # Notebook 2 sets the initial condition from the same value

# close the output files from the prior run before ParFlow overwrites them
pf_out.close()
clm_out.close()

#re-run the model with the new parameters
run.run(working_directory=base, skip_validation=True)

In [ ]:
# reload in output files
file_in = f"{base}/{run_name}.out.00001.nc"
print(file_in)
pf_out = xr.open_mfdataset(file_in)
print(pf_out.head())

# recompute runoff and porosity for the new run
porosity = run.Geom.soil.Porosity.Value
runoff = calc_runoff(pf_out)
runoff_mmh = runoff / dx * 1000.0
print('total runoff [mm]: ', float(runoff_mmh.sum()))

# Plot the soil moisture at the top cell with precip and runoff for our PFCLM run
fig, ax = plt.subplots()
ax2 = ax.twinx()
ax.plot(pf_out['saturation'][:,9,0,0]*100*porosity, label='Soil Moisture 5cm',color='red') 
ax2.plot(runoff, label='Runoff [m2/h]',color='green') 
ax.plot(forcing_df['APCP [mm/s]']*3600, label='Precipitation [mm/h]',color='blue')
ax.set_xlabel('Time [hr]')  
ax.set_ylabel('Water Content and Precipitation')  
ax2.set_ylabel('Runoff') 
plt.legend()
plt.show()